# PCA sul disconnettoma voxelwise, senza rotazione

**Domanda.** Come si distribuisce la variabilità tra i soggetti di `s2.4-vol` quando la si riassume con la sola PCA (covarianza, senza standardizzare, senza varimax)? Quante componenti servono, da cosa dipendono, e cosa si vede se si mettono i soggetti sulle prime due?

**Cosa legge.** La matrice `matrix.npy` e `metadata.csv` della sessione `s2.4-vol` (5845 soggetti × 291.345 voxel) e `assets/metadata/participants.csv` (`lesion_volume_voxels_2mm`, `lesion_side`).

**Cosa scrive.** Solo una cache in `results/pipeline_building/pca_sdc_voxelwise/<sessione>/` (cartella ignorata da git), per non rifare i circa 4 minuti della PCA e i circa 3 delle mappe. Il calcolo sta in `src/analysis/covariance_pca.py`.

**Cosa non fa.** Non ruota le componenti: per il confronto con il paper (varimax, componenti come profili) c'è `thiebaut_2020_replication`. Non fa clustering e non salva nessuna matrice di punteggi.

| Sezione | Contenuto |
|---|---|
| 1 | Scelta della matrice e dei parametri |
| 2 | PCA sulla covarianza |
| 3 | Quanta varianza spiega ogni componente |
| 4 | Quanto ogni componente dipende dal volume della lesione |
| 5 | Componente 1 contro componente 2 |
| 6 | Mappe delle componenti nel cervello |
| 7 | Stabilità delle mappe tra due metà dei soggetti |

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns
from nilearn import plotting

# La radice del repo è la prima cartella, risalendo da quella corrente, che contiene assets/metadata. Non dipende da
# quante volte si esegue la cella: l'os.chdir qui sotto sposta la cartella corrente nella radice.
_start = Path.cwd().resolve()
NOTEBOOK_REPO_ROOT = next((p for p in [_start, *_start.parents] if (p / "assets" / "metadata").is_dir()), None)
if NOTEBOOK_REPO_ROOT is None:
    raise FileNotFoundError(f"nessuna cartella con assets/metadata sopra {_start}: il notebook va aperto dentro il repo")
if str(NOTEBOOK_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_REPO_ROOT))
# load_participants_registry legge assets/metadata/participants.csv con un path relativo alla repo root.
os.chdir(NOTEBOOK_REPO_ROOT)

from src.analysis.covariance_pca import (
    CovariancePca,
    component_t_maps,
    fit_covariance_pca,
    map_reproducibility,
    positive_peak_signs,
    squared_correlation,
)
from src.utils.participants import load_participants_registry

# src.analysis.* imposta matplotlib.use("Agg") al proprio import: senza questo i plot non compaiono.
%matplotlib inline

sns.set_theme(style="whitegrid")

## 1. Scelta della matrice e dei parametri

Questi sono i valori da modificare:

- `SESSION_DIR`: la cartella della matrice. Qui `s2.4-vol`, il disconnettoma voxelwise dei 5845 soggetti ammessi.

- `VARIANCE_SHARE`: la quota di varianza che decide quante componenti guardare nelle sezioni 4, 6 e 7. Con `0.9` sono le prime componenti che insieme spiegano almeno il 90%.

- `LESION_VOLUME_COLUMN`: la colonna del volume della lesione, in voxel a 2 mm.

- `USE_CACHE`: con `True` riusa il calcolo già fatto, con `False` lo rifà e sovrascrive la cache.

In [ ]:
SESSION_DIR = Path("data/derived/sdc_matrix/07-10_s2.4-vol")
VARIANCE_SHARE = 0.9
LESION_VOLUME_COLUMN = "lesion_volume_voxels_2mm"
USE_CACHE = True

CACHE_DIR = Path("results/pipeline_building/pca_sdc_voxelwise") / SESSION_DIR.name

# La matrice resta su disco (mmap): ogni calcolo la legge a blocchi di colonne, mai tutta insieme.
X = np.load(SESSION_DIR / "matrix.npy", mmap_mode="r")
metadata = pd.read_csv(SESSION_DIR / "metadata.csv")
if len(metadata) != X.shape[0]:
    raise ValueError(f"metadata.csv ha {len(metadata)} righe ma la matrice ha {X.shape[0]} soggetti")

print(f"Sessione: {SESSION_DIR.name}")
print(f"Matrice: {X.shape[0]} soggetti x {X.shape[1]} voxel, tipo {X.dtype}")
print("Soggetti per dataset:")
for name, count in metadata["dataset"].value_counts().sort_index().items():
    print(f"  {name}: {count}")

## 2. PCA sulla covarianza

**Cosa fa la PCA.** Trova le direzioni lungo cui i soggetti differiscono di più. La prima componente è la direzione con più variabilità, la seconda è la più variabile tra quelle perpendicolari alla prima, e così via. Per ogni componente si ottengono tre cose: la *varianza* che cattura, i *pesi per voxel* (quali voxel contano) e il *punteggio* di ogni soggetto (la sua posizione su quella direzione).

**Covarianza, senza standardizzare.** Ogni voxel viene centrato (si toglie la sua media sui soggetti) ma non diviso per la sua deviazione standard. Un voxel che varia molto tra i soggetti pesa quindi di più di uno che varia poco.

**Perché non è la PCA di sempre.** La matrice di covarianza tra 291.345 voxel avrebbe 291.345 × 291.345 valori (circa 680 GB). Si ottiene lo stesso risultato dalla matrice 5845 × 5845 delle somiglianze tra soggetti: gli autovalori sono identici, e i test lo verificano contro `sklearn.decomposition.PCA` su matrici piccole.

Il calcolo dura circa 4 minuti la prima volta.

In [ ]:
PCA_CACHE = CACHE_DIR / "pca.npz"
FINGERPRINT = CACHE_DIR / "source.json"

# La cache sa da quale matrice viene: se la matrice cambia, la cache non vale più e il notebook si ferma.
source = {
    "session": SESSION_DIR.name,
    "shape": list(X.shape),
    "matrix_mtime_ns": (SESSION_DIR / "matrix.npy").stat().st_mtime_ns,
}
if USE_CACHE and PCA_CACHE.is_file():
    if json.loads(FINGERPRINT.read_text()) != source:
        raise ValueError(f"la cache in {CACHE_DIR} viene da un'altra matrice: cancella la cartella o metti USE_CACHE = False")
    cached = np.load(PCA_CACHE)
    pca = CovariancePca(cached["eigenvalues"], cached["left_vectors"], cached["column_means"])
    print(f"PCA letta dalla cache: {PCA_CACHE}")
else:
    pca = fit_covariance_pca(X)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    np.savez(PCA_CACHE, eigenvalues=pca.eigenvalues, left_vectors=pca.left_vectors, column_means=pca.column_means)
    FINGERPRINT.write_text(json.dumps(source))
    print(f"PCA calcolata e salvata in {PCA_CACHE}")

N_COMPONENTS = pca.n_components_for(VARIANCE_SHARE)
scores = pca.pca_scores(N_COMPONENTS)

print(f"Componenti con varianza non nulla: {pca.n_nonzero_components()}")
print(f"Varianza totale della matrice: {pca.eigenvalues.sum():.1f}")
print(f"Guardo le prime {N_COMPONENTS} componenti (quelle che insieme spiegano almeno il {VARIANCE_SHARE:.0%})")
print(f"Punteggi: {scores.shape[0]} soggetti x {scores.shape[1]} componenti")

## 3. Quanta varianza spiega ogni componente

La **varianza spiegata** di una componente è la sua quota sulla variabilità totale. Esempio: se la prima spiega il 26% e la seconda il 25%, insieme ne spiegano il 51%, e questa somma è la **varianza cumulata**.

Lo **scree plot** (a sinistra) mostra la quota di ogni componente: dove la curva smette di scendere ripida, le componenti successive aggiungono poco. A destra la varianza cumulata, con le soglie del 50%, 80%, 90%.

In [ ]:
THRESHOLDS = (0.5, 0.8, 0.9)
n_for_share = {share: pca.n_components_for(share) for share in THRESHOLDS}
ratio = pca.explained_variance_ratio
cumulative = pca.cumulative_variance_ratio

for share, n_components in n_for_share.items():
    print(f"Per spiegare almeno il {share:.0%} della varianza servono {n_components} componenti su {pca.n_nonzero_components()}")
print()
for k in range(10):
    print(f"Componente {k + 1}: {ratio[k]:.1%} della varianza, {cumulative[k]:.1%} con le componenti fino a questa")

N_SCREE = 20
N_CURVE = 150
fig, (ax_scree, ax_cumulative) = plt.subplots(1, 2, figsize=(13, 4.5))
ax_scree.bar(np.arange(1, N_SCREE + 1), ratio[:N_SCREE], color="tab:blue")
ax_scree.set_xlabel("Componente")
ax_scree.set_ylabel("Quota di varianza spiegata")
ax_scree.set_title(f"Scree plot (prime {N_SCREE} componenti)")

ax_cumulative.plot(np.arange(1, N_CURVE + 1), cumulative[:N_CURVE], color="tab:blue")
for share, n_components in n_for_share.items():
    ax_cumulative.axhline(share, color="grey", linestyle=":", linewidth=1)
    ax_cumulative.plot([n_components], [cumulative[n_components - 1]], "o", color="tab:red")
    ax_cumulative.annotate(f"{share:.0%}: {n_components} comp.", (n_components, cumulative[n_components - 1]),
                           textcoords="offset points", xytext=(8, -14))
ax_cumulative.set_xlabel("Numero di componenti")
ax_cumulative.set_ylabel("Varianza cumulata")
ax_cumulative.set_title(f"Varianza cumulata (prime {N_CURVE} componenti)")
ax_cumulative.set_ylim(0, 1.02)
plt.show()

## 4. Quanto ogni componente dipende dal volume della lesione

**Cos'è R².** Per ogni componente si guarda quanto il suo punteggio cresce o cala insieme al volume della lesione. R² è il quadrato della correlazione, e si legge come la quota della variabilità del punteggio che si spiega conoscendo il volume.

- R² = 0.04: il volume spiega il 4% della variabilità. La componente è quasi indipendente dalla dimensione della lesione.

- R² = 0.50: il volume spiega la metà. La componente misura soprattutto quanto è grande la lesione.

Il paper riporta R² < 0.1 per tutte le sue componenti (ruotate).

In [ ]:
registry = load_participants_registry().set_index("subject_id")
missing = sorted(set(metadata["subject_id"]) - set(registry.index))
if missing:
    raise ValueError(f"{len(missing)} soggetti della matrice non sono nel registro: {missing[:5]}")
volume = registry.loc[metadata["subject_id"], LESION_VOLUME_COLUMN].to_numpy(dtype=float)
if np.isnan(volume).any():
    raise ValueError(f"{int(np.isnan(volume).sum())} soggetti senza {LESION_VOLUME_COLUMN}")

r2 = squared_correlation(scores, volume)
print(f"Volume della lesione ({LESION_VOLUME_COLUMN}): mediana {np.median(volume):.0f} voxel, da {volume.min():.0f} a {volume.max():.0f}")
print(f"R² massimo {r2.max():.3f}, mediano {np.median(r2):.3f}, {int((r2 > 0.1).sum())} su {len(r2)} componenti sopra 0.1")
print()
for k in np.argsort(r2)[::-1][:5]:
    print(f"Componente {k + 1}: R² con il volume = {r2[k]:.3f}")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(np.arange(1, N_COMPONENTS + 1), r2, color="tab:orange")
ax.axhline(0.1, color="tab:red", linestyle="--", linewidth=1, label="soglia 0.1 del paper")
ax.set_xlabel("Componente")
ax.set_ylabel("R² con il volume della lesione")
ax.set_title(f"{SESSION_DIR.name}: dipendenza dal volume della lesione (PCA, non ruotata)")
ax.legend()
plt.show()

## 5. Componente 1 contro componente 2

Un punto per soggetto, con il punteggio sulla prima componente in orizzontale e sulla seconda in verticale. Le prime due componenti spiegano insieme circa metà della varianza, quindi questo è il piano che mostra più struttura.

Tre colorazioni: il **lato** della lesione, il **dataset** e il **volume** (in scala logaritmica, perché va da pochi voxel a decine di migliaia). I 4 soggetti senza lato sono segnati a parte, non assegnati a un lato.

Se i colori per lato si separano, la lateralizzazione domina la struttura. Se quelli per dataset si separano, ci sono differenze tra coorti (di acquisizione o di popolazione) che un clustering rischia di trovare al posto di quelle biologiche.

In [ ]:
side = registry.loc[metadata["subject_id"], "lesion_side"].fillna("sconosciuto").to_numpy()
side_colors = {"left": "tab:blue", "right": "tab:red", "both": "tab:green", "sconosciuto": "black"}
datasets = metadata["dataset"].to_numpy()
log_volume = np.log10(volume)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharex=True, sharey=True)

for name, color in side_colors.items():
    chosen = side == name
    axes[0].scatter(scores[chosen, 0], scores[chosen, 1], s=5, alpha=0.5, color=color, label=f"{name} ({int(chosen.sum())})")
axes[0].set_title("Lato della lesione")
axes[0].legend(markerscale=3)

palette = sns.color_palette("tab10", n_colors=len(np.unique(datasets)))
for (name, color) in zip(sorted(np.unique(datasets)), palette):
    chosen = datasets == name
    axes[1].scatter(scores[chosen, 0], scores[chosen, 1], s=5, alpha=0.5, color=color, label=f"{name} ({int(chosen.sum())})")
axes[1].set_title("Dataset")
axes[1].legend(markerscale=3, fontsize=8)

points = axes[2].scatter(scores[:, 0], scores[:, 1], s=5, alpha=0.6, c=log_volume, cmap="viridis")
fig.colorbar(points, ax=axes[2], label="log10 del volume (voxel a 2 mm)")
axes[2].set_title("Volume della lesione")

for ax in axes:
    ax.set_xlabel(f"Componente 1 ({ratio[0]:.1%} della varianza)")
axes[0].set_ylabel(f"Componente 2 ({ratio[1]:.1%} della varianza)")
plt.show()

## 6. Mappe delle componenti nel cervello

**Cos'è la mappa di una componente.** Per ogni voxel si guarda quanto il suo valore di disconnessione cresce insieme al punteggio della componente, tenendo conto delle altre componenti (una regressione multipla per voxel). Il risultato per voxel è una statistica *t*: l'effetto diviso per il suo errore.

Esempio: *t* = 10 vuol dire che l'effetto è dieci volte più grande dell'incertezza con cui lo si misura. Con 5845 soggetti anche effetti piccoli hanno *t* alto, quindi la soglia sotto è una frazione del picco di ciascuna componente.

**Segno.** Il segno di una componente è arbitrario. Ogni mappa è orientata in modo che il voxel con *t* più grande in valore assoluto sia positivo.

I valori da modificare:

- `COMPONENTS_TO_PLOT`: i numeri delle componenti da disegnare, da 1 al numero di componenti guardate. Esempi: `[1, 2, 3, 4]`, `list(range(1, 9))`.

- `MAP_THRESHOLD_FRACTION`: la quota del picco di ciascuna componente sotto cui un voxel non viene colorato. Con `0.3` si vedono i voxel con |*t*| di almeno il 30% del massimo di quella componente.

- `SPLIT_SEED`: il seme del sorteggio delle due metà (sezione 7).

Le due metà e le mappe si calcolano in un solo passaggio sulla matrice, circa 3 minuti la prima volta.

In [ ]:
COMPONENTS_TO_PLOT = [1, 2, 3, 4]
MAP_THRESHOLD_FRACTION = 0.3
SPLIT_SEED = 0

if not 0.0 <= MAP_THRESHOLD_FRACTION < 1.0:
    raise ValueError(f"MAP_THRESHOLD_FRACTION deve stare tra 0 e 1 (1 escluso), trovato {MAP_THRESHOLD_FRACTION}")
bad = [k for k in COMPONENTS_TO_PLOT if not 1 <= k <= N_COMPONENTS]
if bad or len(set(COMPONENTS_TO_PLOT)) != len(COMPONENTS_TO_PLOT):
    raise ValueError(f"COMPONENTS_TO_PLOT deve avere numeri distinti tra 1 e {N_COMPONENTS}, trovato {COMPONENTS_TO_PLOT}")

# Le due metà: sorteggio dei soggetti con un seme fisso, così il risultato si ripete.
order = np.random.default_rng(SPLIT_SEED).permutation(X.shape[0])
half_a = np.sort(order[: X.shape[0] // 2])
half_b = np.sort(order[X.shape[0] // 2 :])
print(f"Metà A: {len(half_a)} soggetti, metà B: {len(half_b)} soggetti (seme {SPLIT_SEED})")

MAPS_CACHE = CACHE_DIR / "maps.npz"
MAPS_FINGERPRINT = CACHE_DIR / "maps_source.json"
maps_source = {**source, "n_components": N_COMPONENTS, "split_seed": SPLIT_SEED}

if USE_CACHE and MAPS_CACHE.is_file():
    if json.loads(MAPS_FINGERPRINT.read_text()) != maps_source:
        raise ValueError(f"la cache delle mappe in {CACHE_DIR} ha altri parametri: metti USE_CACHE = False per rifarla")
    cached = np.load(MAPS_CACHE)
    t_full, t_a, t_b = cached["t_full"], cached["t_a"], cached["t_b"]
    n_degenerate = cached["n_degenerate"]
    print(f"Mappe lette dalla cache: {MAPS_CACHE}")
else:
    maps_full, maps_a, maps_b = component_t_maps(X, scores, [np.arange(X.shape[0]), half_a, half_b])
    t_full, t_a, t_b = (m.t.astype(np.float32) for m in (maps_full, maps_a, maps_b))
    n_degenerate = np.array([m.n_degenerate_voxels for m in (maps_full, maps_a, maps_b)])
    np.savez(MAPS_CACHE, t_full=t_full, t_a=t_a, t_b=t_b, n_degenerate=n_degenerate)
    MAPS_FINGERPRINT.write_text(json.dumps(maps_source))
    print(f"Mappe calcolate e salvate in {MAPS_CACHE}")

print(f"Voxel senza varianza residua (t messo a 0): {n_degenerate[0]} su tutti i soggetti, {n_degenerate[1]} nella metà A, {n_degenerate[2]} nella metà B")

# Stessi segni per le due metà e per la mappa completa: la correlazione tra metà non dipende dal segno scelto.
signs = positive_peak_signs(t_full)
t_full, t_a, t_b = (t * signs[:, None] for t in (t_full, t_a, t_b))
reproducibility = map_reproducibility(t_a, t_b)

mask = np.load(SESSION_DIR / "non_constant_mask.npy")
# Geometria delle mappe: la griglia a 2 mm (solo intestazione). Sfondo: il T1 a 1 mm, perché il file T1 a 2 mm ha i dati troncati.
template = nib.load("assets/templates/tpl-MNI152NLin6Asym_res-2_T1w.nii.gz")
background = nib.load("assets/templates/tpl-MNI152NLin6Asym_res-1_T1w.nii.gz")
if mask.size != int(np.prod(template.shape)) or int(mask.sum()) != X.shape[1]:
    raise ValueError(f"la maschera ({mask.size} voxel, {int(mask.sum())} veri) non corrisponde al template {template.shape} o alla matrice ({X.shape[1]} colonne)")


def map_to_image(values):
    # Le colonne di X sono i voxel della maschera, nell'ordine in cui la griglia è stata srotolata (C-order).
    volume_3d = np.zeros(mask.size, dtype=np.float32)
    volume_3d[mask] = values
    return nib.Nifti1Image(volume_3d.reshape(template.shape), template.affine)


for k in COMPONENTS_TO_PLOT:
    t_map = t_full[k - 1]
    peak_column = int(np.argmax(np.abs(t_map)))
    peak_voxel = np.unravel_index(np.flatnonzero(mask)[peak_column], template.shape)
    peak_mm = nib.affines.apply_affine(template.affine, peak_voxel)
    plotting.plot_stat_map(
        map_to_image(t_map),
        bg_img=background,
        threshold=MAP_THRESHOLD_FRACTION * abs(t_map[peak_column]),
        cut_coords=tuple(peak_mm),
        display_mode="ortho",
        title=f"Componente {k} (PCA): {ratio[k - 1]:.1%} della varianza, t massimo {t_map[peak_column]:.0f}",
    )
    plt.show()

## 7. Stabilità delle mappe tra due metà dei soggetti

I soggetti si dividono a caso in due metà (seme `SPLIT_SEED`). Si calcola la mappa di ogni componente in ciascuna metà, e si correla (Pearson, sui voxel) la mappa della prima con quella della seconda. Vale 1 se le due metà danno la stessa mappa, 0 se non hanno niente in comune.

I punteggi sono quelli calcolati su tutti i soggetti: le due metà non rifanno la PCA, mostrano se la stessa componente si ritrova.

Il paper riporta 0.813 ± 0.079 per le sue componenti ruotate; qui non c'è nessun numero equivalente per la PCA non ruotata, quindi il confronto è solo indicativo.

In [ ]:
print(f"Stabilità tra le due metà, media {reproducibility.mean():.3f} ± {reproducibility.std(ddof=1):.3f}")
print(f"Da {reproducibility.min():.3f} (componente {reproducibility.argmin() + 1}) a {reproducibility.max():.3f} (componente {reproducibility.argmax() + 1})")
print()
for k in range(min(N_COMPONENTS, 10)):
    print(f"Componente {k + 1}: stabilità = {reproducibility[k]:.3f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(np.arange(1, N_COMPONENTS + 1), reproducibility, color="tab:blue")
ax.set_xlabel("Componente")
ax.set_ylabel("Stabilità tra le due metà")
ax.set_title(f"{SESSION_DIR.name}: stabilità delle mappe (PCA, non ruotata)")
ax.set_ylim(0, 1.02)
plt.show()